# Somo la 09 - Mfumo wa Ubunifu wa Metakognitini


## Usanidi

Daftari hili linaonyesha muundo wa Metacognition kwa kutumia Microsoft Agent Framework.

**Mahitaji:**
- Usanidi wa Azure OpenAI umewekwa kupitia mabadiliko ya mazingira
- Azure CLI imethibitishwa (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Metacognition ni Nini?

Metacognition ni **kufikiria kuhusu kufikiri**. Katika muktadha wa maajenti wa AI, inamaanisha kujenga maajenti ambao wanaweza:

- **Kujitafakari** juu ya matokeo yao na mchakato wa kufikiri
- **Kubaini makosa** na kupona kwa ustadi badala ya kushindwa kimya kimya
- **Kutathmini** kama majibu yao ni kamili na ya msaada
- **Kubadilisha mkakati** wakati mbinu ya awali haifanyi kazi (mfano, kurejea kwa mfumo wa akiba)

Mwakilishi mwenye metacognitive haiji jibu tu maswali — huangalia utendaji wake mwenyewe na hubadilika papo hapo.


## Zana za Kuu na Mbadala

Mifumo ya kawaida ya metakujua ni **mkakati wa kuepuka hitilafu**. Wakala hujaribu kwanza zana ya msingi; ikiwa inashindwa (kwa mfano, kosa la 404), wakala hutambua kushindwa na kubadilisha kwa uwazi kwenda kwa zana mbadala.

Hii inaonyesha mifumo halisi ambapo huduma za msingi zinaweza kuwa hazipatikani na mawakala lazima waweke utambuzi wa tatizo kabla ya kuchagua njia mbadala.

Hapo chini tunaeleza zana mbili za kutafuta ndege:
- **Msingi** — inahusu Paris, Tokyo, na Barcelona
- **Mbadala** — inahusu Berlin, Sydney, na New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Wakala Anayejitathmini kwa Kujirekebisha Makosa

Wakala chini ameelekezwa kujaribu kwanza mfumo mkuu wa ndege, kutambua makosa, na kwa uwazi kurudi kwenye mfumo wa ziada. Baada ya kila jibu linajitathmini mfupi kama lilikamilisha kujibu swali la mtumiaji.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Mfano wa Kujitathmini

Sehemu nyingine ya metakognisheni ni **kujitathmini**: wakala tofauti (au wakala huyo yule katika kupitia ya pili) hurejea jibu kwa ukamilifu, usahihi, na msaada.

Chini tunaunda wakala `ResponseEvaluator` anayepima majibu ya mawakala wa usafiri kwa vipimo vitatu.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Muhtasari

Katika somo hili ulijifunza jinsi ya kujenga **mawakala wa metakognitiki** kwa kutumia Microsoft Agent Framework:

- **Kujitathmini**: Mawakala wanaojiangalia wenyewe kwa kufuatilia mantiki yao na kuwasilisha kwa uwazi kilichotokea.
- **Kupona makosa kwa mbadala**: Mwelekeo wa kifaa kikuu + chelezo ambapo wakala hutambua makosa (km, makosa 404) na kwa moja kwa moja anajaribu chanzo mbadala.
- **Kujipima**: Wakala mtathmini mwengine anayepima majibu kwa ukamilifu, usahihi, na msaada.

Mwelekeo huu hufanya mawakala kuwa imara zaidi, wazi, na wa kuaminika — sifa muhimu kwa utekelezaji wa uzalishaji.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
